# Data access and analysis of Metop-A/B/C Level 3 trace gas products relevant for fires

Julian Meyer-Arnek (German Aerospace Center, DLR-DFD)

**Abstract**:

Here it is demonstrated how to 
 - identify usable data collections,
 - connect to a STAC service endpoint,
 - search for available datasets using STAC (discovery),
 - download datasets into a locally available object (XArray),
 - perform visualization and analysis.

This Jupyter Notebook was created for the "EUMeTrain Event Week on Forest Fires - June 2026" (https://www.eumetrain.org/event-weeks/event-week-forest-fires-june-2026)


## Python imports

In [ ]:
from pystac_client import Client

In [ ]:
from odc.stac import stac_load
import xarray as xr

In [ ]:
import numpy

In [ ]:
import math

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt

In [ ]:
import cartopy

## STAC settings

To identify the relevant STAC collection, visit [https://geoservice.dlr.de/eoc/ogc/stac/v1](https://geoservice.dlr.de/eoc/ogc/stac/v1) with your browser.

There are many more STAC catalogues available: just visit [https://stacindex.org](https://stacindex.org) to find more.

Find more about STAC and its specifications here: [https://stacspec.org](https://stacspec.org).


In [ ]:
# STAC endpoint
stacapi_endpoint = "https://geoservice.dlr.de/eoc/ogc/stac/v1/"


### STAC API-Hierarchy

STAC organizes the metadata in an hierarchical way:

- **Collections**: Group of similar products (timeseries)
  - **Items**: Individual (daily) products
    - **Assets**: https-Links to Datasets
   
The STAC API offers a human-readable HTML-version... (see below)

![Visual representation of STAC collection](./Screenshot_STAC_Collection.png)

... as well as a machine-to-machine version with JSON-objects:
```
{"type":"Collection","stac_version":"1.0.0","stac_extensions":["https://stac-extensions.github.io/processing/v1.0.0/schema.json","https://stac-extensions.github.io/datacube/v2.0.0/schema.json","https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"],"id":"METOP_GOME2_L3_P1D_NO2TROPO","title":"MetOp-A/B/C GOME-2 L3 P1D Tropspheric Nitrogen Dioxide (NO2Tropo)","description":"MetOp-A/B/C GOME-2 L3 P1D data of Tropspheric Nitrogen Dioxide (NO2Tropo) with a global coverage.","crs":["http://www.opengis.net/def/crs/OGC/1.3/CRS84"],"keywords":["EOC","Atmos","GOME-2","MetOp","M....
```

In [ ]:
# Now we make some mission specific settings
collections = ["METOP_GOME2_L3_P1D_NO2TROPO"]
platform_b = "MetOp-B"
platform_c = "MetOp-C"
resolution=0.25 # original product resolution


In [ ]:
# Download the complete year
dates_from_to = ["2025-01-01", "2025-12-31"]


In [ ]:
# Area of interest is Iberian Pensinsula
bbox = [-60, 0, 30, 60] # W/S/E/N


<div class="alert alert-block alert-success">
First interaction with STAC: Connect to the STAC catalog.

This is just a "connect", no data is going to be downloaded.
</div>

In [ ]:
catalog = Client.open(
    url=stacapi_endpoint
)

## Identify (discover) available datasets
<div class="alert alert-block alert-success">
Data discovery (catalog search) is performed according to our settings
</div>

In [ ]:
%%time

filter="platform='" + platform_c + "'"
stac_items_metopc = catalog.search(
    collections=collections, 
    datetime=dates_from_to, 
    method="GET", 
    filter=filter,
    max_items=1000
).item_collection()

In [ ]:
stac_items_metopc

*--> Now we identified the relevant datasets which we need for our research.*

## Download discovered datasets to local storage (data cube)

<div class="alert alert-block alert-success">
Now the data of the identifed data sets is going to be downloaded to the local client.

You can concentrate on your work. All ugly work such as 
- re-projection or
- area-slicing 
 
is done on the server.</div>

In [ ]:
%%time

no2trop_metopc = stac_load(
    stac_items_metopc,
    bands=["no2trop_c"],
    crs="EPSG:4326",
    resolution=resolution,
    lon=(bbox[0], bbox[2]),
    lat=(bbox[1], bbox[3]),)

print ("Data is extracted on the server and downloaded to local storage.")

In [ ]:
no2trop_metopc = no2trop_metopc.where(no2trop_metopc <= 1e36, other=numpy.nan)

## Representation of data cube (XArray object)

<div class="alert alert-block alert-success">
Now the data of the identifed data sets is downloaded to the local client and is accessible as XArray object.
</div>

In [ ]:
no2trop_metopc

## Visualize data cube contents (XArray object)

<div class="alert alert-block alert-success">
Now Metop-C data of tropospheric NO2 is time-averaged and visualized.
</div>

In [ ]:
no2trop_metopc["wildfirePeriod"] = no2trop_metopc["no2trop_c"].sel(time=slice("2025-08-12", "2025-08-21")).mean(dim="time")

In [ ]:
colors = [(1,1,1,0.1),     # fully transparent white          
          (1,0,1,0.7),     # semi-transparent violet
          (0.2,0,1,0.9),   # blue
          (1,0,0,1.0),     # red
          (0.8,0,0,1.0)]   # dark red

mycmap = LinearSegmentedColormap.from_list('mycmap2', colors, N=256)

In [ ]:
%%time

proj = cartopy.crs.PlateCarree()

fig, ax = plt.subplots(subplot_kw={"projection": proj}, figsize=(10,8))

no2trop_metopc["wildfirePeriod"].plot (
    ax=ax, 
    levels=[1e14,1e15,2e15,3e15,4e15,5e15,6e15,7e15,8e15,1e16],
    cmap=mycmap) # LinearSegmentedColormap.from_list("", ["white", "violet", "red","blue"]))

ax.add_feature(cartopy.feature.LAND, facecolor='#CCCCCC', edgecolor='#000000')
# ax.add_feature(cartopy.feature.COASTLINE.with_scale('110m'))
ax.add_feature(cartopy.feature.BORDERS, linestyle=':')
ax.set_extent([-20, 10, 20, 60]) # W/E/S/N
ax.set_title ("Tropospheric NO$_2$ (Metop C) during Wildfires [molec/cm$^2$]")

## Compute Difference between Wild Fire Episode and Average Conditions

<div class="alert alert-block alert-success">
In the above viewgraph we see enhanced tropospheric NO$_2$ columns over industrial areas as well over wildfire-affected areas (Iberian Peninsula). To better identify the wildfires, we substract the average NO$_2$ conditions (April to July) from the conditions during the wildfire period (11.-21.08.2025):

tropNO$_2$ (wildfireInduced) = tropNO$_2$ (wildfirePeriod) - tropNO$_2$ (average)
</div>

![Derive wildfire induced tropospheric NO2 by substracting average tropNO2 conditions](./Time-Axis.drawio.png)

In [ ]:
no2trop_metopc["average"] = no2trop_metopc["no2trop_c"].sel(time=slice("2025-04-01", "2025-08-11")).mean(dim="time")

In [ ]:
no2trop_metopc["wildfireInduced"] = no2trop_metopc["wildfirePeriod"] - no2trop_metopc["average"]

In [ ]:
no2trop_metopc

In [ ]:
%%time

proj = cartopy.crs.PlateCarree()

fig, ax = plt.subplots(subplot_kw={"projection": proj}, figsize=(10,8))

no2trop_metopc["wildfireInduced"].plot (
    ax=ax, 
    levels=[1e14,1e15,2e15,3e15,4e15,5e15,6e15,7e15,8e15,1e16],
    cmap=mycmap) # LinearSegmentedColormap.from_list("", ["white", "violet", "red","blue"]))

ax.add_feature(cartopy.feature.LAND, facecolor='#CCCCCC', edgecolor='#000000')
# ax.add_feature(cartopy.feature.COASTLINE.with_scale('110m'))
ax.add_feature(cartopy.feature.BORDERS, linestyle=':')
ax.set_extent([-20, 10, 20, 60]) # W/E/S/N
ax.set_title ("Wildfire-Induced Tropospheric NO$_2$ (Metop C) [molec/cm²]")


In [ ]:
# End of Notebook